# Insider purchase backtest

Do stocks outperform after insiders buy their own company's shares on the open market?

**Pipeline**
1. `sec.py` — download the SEC's quarterly insider data sets and join `NONDERIV_TRANS` + `SUBMISSION` + `REPORTINGOWNER` into one row per open-market purchase (code `P`).
2. `prices.py` — daily prices from Yahoo, cached in `data/prices/`.
3. `backtest.py` — filters → signals → event-study backtest vs a benchmark.

**Point-in-time rule:** every signal enters at the **next trading day's open after `filing_date`** — the day the public could first see the trade — never on the insider's own trade date.

The logic lives in the `.py` modules; this notebook only calls them. Before trusting any number: *Kernel → Restart & Run All*.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

import backtest as bt
import prices
import sec
from backtest import Filters, replace

pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.max_colwidth", 60)
DATA = Path("data")

# The SEC blocks requests without contact info in the User-Agent.
if "SEC_USER_AGENT" not in os.environ:   # a statement, so the value (your email) is never displayed
    os.environ["SEC_USER_AGENT"] = "signal-research YOUR_EMAIL_HERE"

## 1. Config
Data exists from `2006q1`. More quarters = more tickers whose prices must be downloaded (the first run of section 3 is the slow part; after that everything is cached).

In [ ]:
START, END = "2021q1", "2026q2"
PRICE_START = "2020-06-01"   # a bit before START so early signals have price history
BENCHMARK = "SPY"            # try "IWM" too: insider buying skews to small caps
COST_BPS = 50                # round-trip trading cost; small caps deserve more
HOLD = 63                    # trading days shown in charts/tables: 21 ≈ 1m, 63 ≈ 3m, 126 ≈ 6m, 252 ≈ 1y

## 2. SEC data → one row per purchase

In [30]:
quarters = sec.download_quarters(sec.quarter_range(START, END), DATA / "sec")
sec.build_purchases(quarters, DATA / "sec", DATA / "purchases.parquet")
purchases = sec.load_purchases(DATA / "purchases.parquet")
print(f"{len(purchases):,} open-market purchases across {purchases.ticker.nunique():,} tickers")
purchases.head()

132,266 open-market purchases across 5,792 tickers


,accession,trans_idx,issuer_cik,issuer_name,ticker,filing_date,trans_date,security_title,shares,price,...,is_late,owner_names,owner_ciks,owner_titles,n_owners,is_officer,is_director,is_ten_pct_owner,is_entity,is_ceo_cfo
0,0000002230-21-000001,1,0000002230,"ADAMS DIVERSIFIED EQUITY FUND, INC.",ADX,2021-01-04,2020-12-31,Common Stock,"2,895.0000",17.2700,...,False,JAMMET MARY CHRIS,0001599796,NaN,1,False,True,False,False,False
1,0000216851-21-000001,1,0000216851,"ADAMS NATURAL RESOURCES FUND, INC.",PEO,2021-01-04,2020-12-31,Common Stock,"4,385.0000",11.4000,...,False,JAMMET MARY CHRIS,0001599796,NaN,1,False,True,False,False,False
2,0000898432-21-000010,1,0001589390,Evanston Alternative Opportunities Fund,NaN,2021-01-04,2021-01-01,Class I common shares of beneficial interest,"270,258.1000",11.1000,...,False,Moyer Robert F.,0001600747,NaN,1,False,False,False,False,False
3,0000899243-21-000053,1,0001181848,SKYBRIDGE MULTI-ADVISER HEDGE FUND PORTFOLIOS LLC,NaN,2021-01-04,2021-01-01,Common Stock,47.3600,"1,055.6700",...,False,Scaramucci Anthony,0001555327,NaN,1,False,False,False,False,False
4,0000899243-21-000160,1,0001656634,"Gritstone Oncology, Inc.",GRTS,2021-01-04,2020-12-30,Common Stock,"1,347,709.0000",3.7100,...,False,Woiwode Thomas,0001619294,NaN,1,False,True,False,False,False


In [31]:
purchases.groupby(purchases.filing_date.dt.year).agg(
    purchases=("accession", "size"),
    median_usd=("value_usd", "median"),
    officers=("is_officer", "sum"),
    directors=("is_director", "sum"),
    entities=("is_entity", "sum"),
)

,purchases,median_usd,officers,directors,entities
filing_date,,,,,
2021,24591,"19,712.0000",8528,16473,2094
2022,32212,"19,676.6600",11825,21265,2540
2023,24604,"17,200.0000",9455,15984,1427
2024,20659,"15,730.0000",7720,13148,1607
2025,20698,"19,946.0000",9067,12894,1977
2026,9502,"34,053.7500",3723,5365,1243


The largest "purchases" are where data errors hide (unit mix-ups, funds holding board seats). The filters' `max_usd` cap and `exclude_entities` deal with most of them — worth eyeballing anyway.

In [32]:
purchases.nlargest(10, "value_usd")[
    ["ticker", "issuer_name", "filing_date", "owner_names", "owner_titles", "value_usd", "is_entity"]
]

,ticker,issuer_name,filing_date,owner_names,owner_titles,value_usd,is_entity
85761,REEMF,RARE ELEMENT RESOURCES LTD,2024-03-19,Synchron,NaN,"7,166,800,521,173,730.0000",False
12370,ASTI,"Ascent Solar Technologies, Inc.",2021-08-04,"BD1 Investment Holding, LLC; Kuhn Johannes; Kuhn Ute",NaN,"6,666,666,773,333,334.0000",False
10747,GOBI,Gobi Acquisition Corp.,2021-07-06,PAG Ltd,NaN,"4,000,000,000,000,000.0000",True
10748,GOBI,Gobi Acquisition Corp.,2021-07-06,PAG Investment LP,NaN,"4,000,000,000,000,000.0000",True
10749,GOBI,Gobi Acquisition Corp.,2021-07-06,PAG Capital II Ltd,NaN,"4,000,000,000,000,000.0000",True
126363,REEMF,RARE ELEMENT RESOURCES LTD,2026-03-13,BLUE JAMES N,NaN,"2,407,160,212,532,064.0000",False
126971,REEMF,RARE ELEMENT RESOURCES LTD,2026-03-23,GENERAL ATOMIC TECHNOLOGIES CORP; Tenaya Corp; Synchron,NaN,"2,407,160,212,532,064.0000",False
115838,HYEX,HEALTHY EXTRACTS INC.,2025-08-22,Swanson Donald William,President,"307,763,430,923,520.0000",False
26435,SIMP,"Simply, Inc.",2022-01-31,House of Lithium LLC; SOL Global Investments Corp.; SOL ...,NaN,"13,968,151,947,308.0000",True
96922,OPGN,OPGEN INC,2024-10-08,AEI Capital Ltd,NaN,"9,808,164,000,000.0000",True


## 3. Prices
First run downloads every ticker (can take a while for many years); reruns only fetch new ones. `add_price_check` then compares each reported purchase price with that day's market close to flag unit/currency errors.

In [33]:
tickers = list(purchases.ticker.dropna().unique()) + [BENCHMARK]
prices.fetch_prices(tickers, PRICE_START, DATA / "prices")
px = prices.load_prices(tickers, DATA / "prices")
purchases = bt.add_price_check(purchases, px)
purchases.price_ok.value_counts(dropna=False).rename({None: "no price data"})

0 tickers to download
done; 0 tickers had no data on Yahoo


price_ok
True             94899
no price data    35576
False             1791
Name: count, dtype: int64

### Company size at the time of each purchase
`market_cap` = latest shares outstanding reported to the SEC (10-K/10-Q cover page) × the insider's purchase price. Both are as of the trade, so it isn't distorted by later splits, and it avoids the look-ahead bias of using today's list of large caps. Rows with unknown size are dropped whenever a size filter is set.

In [ ]:
shares_out = sec.fetch_shares_outstanding(int(START[:4]) - 1, int(END[:4]), DATA / "shares_outstanding.parquet")
purchases = bt.add_market_cap(purchases, shares_out)
print(f"market cap known for {purchases.market_cap.notna().mean():.0%} of purchases")
pd.cut(purchases.market_cap, [0, 300e6, 2e9, 10e9, 200e9, float("inf")],
       labels=["micro <300M", "small 300M-2B", "mid 2-10B", "large 10-200B", "mega >200B"]).value_counts(dropna=False).sort_index()

## 4. Baseline backtest
Default `Filters()`: officers + directors, no funds/companies, no 10b5-1 plan trades, passes the price check, ≤ $500M per filing. One signal per ticker per filing date, with a 30-day cooldown so one buying spree isn't counted as many bets.

Returns are after `COST_BPS`; **excess** = stock return − benchmark return over the same dates.

In [34]:
base = Filters()
signals = bt.build_signals(bt.apply_filters(purchases, base))
result = bt.run_backtest(signals, px, benchmark=BENCHMARK, cost_bps=COST_BPS)
print(result.coverage())
result.summary()

22668 signals, 4618 (20%) had no price data -> those are missing from the results (survivorship bias risk)


,n,mean_ret,mean_excess,median_excess,hit_rate,t_stat,delisted_exits
hold_days,,,,,,,
21,18013,0.0132,0.0018,-0.0137,0.4342,1.0934,145
63,17964,0.0403,-0.0018,-0.0334,0.4166,-0.3688,153
126,16950,0.0530,-0.0248,-0.0653,0.3839,-5.3856,153
252,15084,0.1160,-0.0347,-0.1227,0.3606,-3.9372,153


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
yearly = result.by_year(HOLD)
yearly["mean_excess"].plot.bar(ax=axes[0], title=f"Mean {HOLD}-day excess return by entry year")
axes[0].axhline(0, color="grey", lw=0.8)
axes[0].yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")

result.events[f"excess_{HOLD}"].clip(-1, 2).plot.hist(ax=axes[1], bins=80, title=f"{HOLD}-day excess return per signal")
axes[1].axvline(0, color="grey", lw=0.8)
axes[1].xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
plt.tight_layout()

### Large caps only (≥ $10B)
Heavily analysed companies, far fewer missing prices — the most trustworthy slice of the data. Change `min_market_cap` to try other cut-offs.

In [ ]:
large = replace(base, min_market_cap=10e9)
result_lc = bt.run_backtest(bt.build_signals(bt.apply_filters(purchases, large)), px, benchmark=BENCHMARK, cost_bps=COST_BPS)
print(result_lc.coverage())
result_lc.summary()

In [ ]:
result_lc.by_year(HOLD)

Mean vs **median** matters here: a few huge winners can drag the mean up while the typical signal loses. Check `hit_rate` and `median_excess` before getting excited.

## 5. Compare filter variants

In [ ]:
variants = {
    "baseline": base,
    ">= $100k": replace(base, min_usd=100_000),
    ">= $500k": replace(base, min_usd=500_000),
    "stake +20%": replace(base, min_stake_increase=0.2),
    "cluster >= 2": replace(base, min_cluster=2),
    "cluster >= 3": replace(base, min_cluster=3),
    "CEO/CFO only": replace(base, ceo_cfo_only=True),
    "+ 10% owners & entities": replace(base, ten_pct_owners=True, exclude_entities=False),
    "micro < $300M": replace(base, max_market_cap=300e6),
    "small $300M-2B": replace(base, min_market_cap=300e6, max_market_cap=2e9),
    "mid $2-10B": replace(base, min_market_cap=2e9, max_market_cap=10e9),
    "large >= $10B": replace(base, min_market_cap=10e9),
}
bt.compare(variants, purchases, px, hold_days=HOLD, benchmark=BENCHMARK, cost_bps=COST_BPS)

## 5b. Sharpe ratio vs the S&P 500
Simulates each variant as a portfolio: the same amount goes into every signal at the next open after filing, held `HOLD` trading days, positions drift with their own price. Idle cash earns the 13-week T-bill rate (`^IRX`). Everything is measured over the same days as buying and holding `BENCHMARK`.

- **sharpe**: return above T-bills per unit of volatility (annualised). Compare with the benchmark row.
- **info_ratio**: return above the benchmark per unit of deviation from it. Positive = beat it on a risk-adjusted basis.
- **beta**: sensitivity to the benchmark; **max_drawdown**: worst peak-to-trough loss.

In [ ]:
prices.fetch_prices(["^IRX"], PRICE_START, DATA / "prices")
rf_daily = prices.load_prices(["^IRX"], DATA / "prices")["^IRX"]["Close"] / 100 / 252   # yield % -> daily rate

sharpe_stats, daily_returns = bt.compare_sharpe(variants, purchases, px, benchmark=BENCHMARK,
                                                hold_days=HOLD, cost_bps=COST_BPS, rf_daily=rf_daily)
sharpe_stats.sort_values("sharpe", ascending=False)

In [ ]:
show = [BENCHMARK, "baseline", "micro < $300M", "small $300M-2B", "mid $2-10B", "large >= $10B"]
ax = (1 + daily_returns[show]).cumprod().plot(figsize=(12, 5), logy=True,
                                            title=f"Growth of $1 ({HOLD}-day holds, {COST_BPS} bps costs)")
ax.set_ylabel("value of $1 (log scale)")

## 6. Out-of-sample check
Trying many variants and keeping the best one overfits. Pick your variant using **only the earlier period**, then look at the later period exactly once.

In [ ]:
SPLIT = "2024-01-01"   # leave enough years on both sides of the split
train = purchases[purchases.filing_date < SPLIT]
test = purchases[purchases.filing_date >= SPLIT]
bt.compare(variants, train, px, hold_days=HOLD, benchmark=BENCHMARK, cost_bps=COST_BPS)

In [ ]:
chosen = "cluster >= 2"   # <- decide from the train table above, then run this once
bt.compare({chosen: variants[chosen]}, test, px, hold_days=HOLD, benchmark=BENCHMARK, cost_bps=COST_BPS)

## 7. Look at individual signals

In [ ]:
cols = ["ticker", "issuer_name", "filing_date", "entry_date", "value_usd", "n_insiders_window", "owners", f"excess_{HOLD}"]
result.events.sort_values(f"excess_{HOLD}", ascending=False)[cols].head(10)

In [ ]:
result.events.sort_values(f"excess_{HOLD}")[cols].head(10)

## Before trusting a result
- **Survivorship:** check `result.coverage()`. Signals without Yahoo prices (often later-delisted stocks) are missing, which flatters results. Confirm promising findings with delisted-inclusive data (e.g. Sharadar) before risking money.
- **Overlapping holds:** `t_stat` treats signals as independent; they overlap in time, so real significance is lower.
- **10b5-1:** the plan flag only exists on filings from 2023; earlier pre-planned trades are still in the data.
- **Costs:** 50 bps round trip is optimistic for micro caps. Try `COST_BPS = 150` and see what survives.
- **Benchmark:** insider buys skew small-cap; compare against `IWM` as well as `SPY`.